# Đồ Án 2 — Data Fitting và Phương Pháp OLS
### Phần 1: Hàm 1 - 4

---


| # | Hàm | Mô tả |
|---|-----|-------|
| 1 | `ols_fit` | Ước lượng hệ số $\hat{\beta}$ và phương sai nhiễu $\hat{\sigma}^2$ |
| 2 | `hat_matrix` | Tính hat matrix $H$ và kiểm tra tính idempotent |
| 3 | `model_metrics` | Tính RSS, TSS, $R^2$, $\bar{R}^2$, F-statistic |
| 4 | `coef_inference` | Kiểm định hệ số: SE, t-stat, p-value, khoảng tin cậy |

## 0. Import và Dữ Liệu Kiểm Thử

In [ ]:
import math
import numpy as np
from scipy import stats
from sklearn.linear_model import LinearRegression

import sys
sys.path.insert(0, '.')
from ols_implementation import (
    ols_fit, hat_matrix, model_metrics, coef_inference,
    print_results, print_hat_matrix, print_metrics, print_inference
)

print("Import thành công!")

In [ ]:

X1 = [[1.0], [3.0], [4.0], [7.0], [9.0], [12.0]]
y1 = [0.0,   2.0,   5.0,  10.0,  12.0,   16.0]

print("Dữ liệu đã sẵn sàng.")

---
## Hàm 1 — `ols_fit`: Ước Lượng OLS

### 1.1 Mô hình tuyến tính

Giả sử dữ liệu tuân theo mô hình:

$$
\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon},
\qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0},\, \sigma^2 \mathbf{I}_n)
$$

trong đó $\mathbf{X} \in \mathbb{R}^{n \times (p+1)}$ là **design matrix** (cột đầu toàn số 1 cho intercept), $\boldsymbol{\beta} \in \mathbb{R}^{p+1}$ là vector tham số cần ước lượng.

### 1.2 Hàm mất mát RSS

OLS tìm $\hat{\boldsymbol{\beta}}$ tối thiểu hoá **Residual Sum of Squares**:

$$
\text{RSS}(\boldsymbol{\beta})
= \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|_2^2
= \sum_{i=1}^{n}(y_i - \mathbf{x}_i^\top \boldsymbol{\beta})^2
$$

### 1.3 Chứng minh nghiệm (Normal Equations)

Tính gradient và đặt bằng 0:

$$
\nabla_{\boldsymbol{\beta}}\,\text{RSS}
= -2\mathbf{X}^\top(\mathbf{y} - \mathbf{X}\boldsymbol{\beta}) = \mathbf{0}
\;\Longrightarrow\;
\mathbf{X}^\top\mathbf{X}\,\boldsymbol{\beta} = \mathbf{X}^\top\mathbf{y}
$$

Khi $\mathbf{X}^\top\mathbf{X}$ khả nghịch (rank đầy đủ), nghiệm duy nhất là:

$$
\boxed{\hat{\boldsymbol{\beta}}_{\text{OLS}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}}
$$

### 1.4 Ước lượng phương sai nhiễu

Ước lượng không chệch của $\sigma^2$:

$$
\boxed{\hat{\sigma}^2 = \frac{\text{RSS}}{n - p - 1}
= \frac{\|\mathbf{y} - \mathbf{X}\hat{\boldsymbol{\beta}}\|^2}{n - p - 1}}
$$

Mẫu số $n - p - 1$ là bậc tự do còn lại (đã dùng $p+1$ tham số). Chia cho $n-p-1$ thay vì $n$ để đảm bảo $\mathbb{E}[\hat{\sigma}^2] = \sigma^2$.

In [ ]:

result1 = ols_fit(X1, y1)
print_results(result1, feature_names=["x"])
print(f"  n = {result1['n']},  p = {result1['p']},  dof = {result1['dof']}")

In [ ]:

X_np = np.array(X1)
y_np = np.array(y1)
X_design = np.hstack([np.ones((len(X_np), 1)), X_np])


beta_np, _, _, _ = np.linalg.lstsq(X_design, y_np, rcond=None)
print("NumPy lstsq  beta:", beta_np)


lr = LinearRegression().fit(X_np, y_np)
print("sklearn      beta:", [lr.intercept_] + list(lr.coef_))

print("Cai dat cua ta beta:", result1['beta_hat'])
print("Khop NumPy (beta):", np.allclose(result1['beta_hat'], beta_np, atol=1e-4))

---
## Hàm 2 — `hat_matrix`: Ma Trận Chiếu

### 2.1 Định nghĩa

**Hat matrix** (projection matrix) là:

$$
\boxed{H = \mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top \in \mathbb{R}^{n \times n}}
$$

Tên gọi xuất phát từ quan hệ: $\hat{\mathbf{y}} = H\mathbf{y}$ — $H$ "đặt mũ" lên $\mathbf{y}$.

### 2.2 Các tính chất quan trọng

| Tính chất | Công thức | Ý nghĩa |
|-----------|-----------|--------|
| **Idempotent** | $H^2 = H$ | Chiếu hai lần = chiếu một lần |
| **Đối xứng** | $H^\top = H$ | Ma trận chiếu trực giao |
| **Fitted values** | $\hat{\mathbf{y}} = H\mathbf{y}$ | Chiếu $\mathbf{y}$ lên $\text{col}(\mathbf{X})$ |
| **Residuals** | $\hat{\boldsymbol{\varepsilon}} = (I-H)\mathbf{y}$ | Phần thẳng góc với $\text{col}(\mathbf{X})$ |
| **Rank** | $\text{rank}(H) = p+1$ | Số chiều của không gian con |

### 2.3 Chứng minh tính idempotent

$$
H^2 = \bigl[\mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\bigr]
      \bigl[\mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\bigr]
= \mathbf{X}\underbrace{(\mathbf{X}^\top\mathbf{X})^{-1}(\mathbf{X}^\top\mathbf{X})}_{=\,I_{p+1}}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top
= \mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top = H \quad \blacksquare
$$

In [ ]:

h_result = hat_matrix(X1)
print_hat_matrix(h_result)

In [ ]:

X_d = np.hstack([np.ones((len(X1), 1)), np.array(X1)])
H_np = X_d @ np.linalg.inv(X_d.T @ X_d) @ X_d.T
H_our = np.array(h_result['H'])

print("Max absolute difference (H_numpy vs H_ours):",
      np.max(np.abs(H_np - H_our)))

print("H^2 = H (NumPy check):",
      np.allclose(H_np @ H_np, H_np))

print("H = H^T (symmetry)  :",
      np.allclose(H_np, H_np.T))

print(f"trace(H) = {np.trace(H_np):.4f}  (phai = {len(X1[0])+1})")
print("Khop NumPy (H):", np.allclose(H_our, H_np, atol=1e-4))

---
## Hàm 3 — `model_metrics`: Đánh Giá Mô Hình

### 3.1 Phân rã tổng bình phương (ANOVA)

$$
\underbrace{\sum_{i=1}^n (y_i - \bar{y})^2}_{\text{TSS}}
= \underbrace{\sum_{i=1}^n (\hat{y}_i - \bar{y})^2}_{\text{MSS (explained)}}
+ \underbrace{\sum_{i=1}^n (y_i - \hat{y}_i)^2}_{\text{RSS (unexplained)}}
$$

### 3.2 Hệ số xác định $R^2$

$$
\boxed{R^2 = 1 - \frac{\text{RSS}}{\text{TSS}} \in [0, 1]}
$$

**Giải thích:** $R^2$ đo tỉ lệ biến động của $y$ được giải thích bởi mô hình.  
- $R^2 = 1$: mô hình khớp hoàn hảo  
- $R^2 = 0$: mô hình không tốt hơn dự đoán bằng $\bar{y}$

**Lưu ý:** $R^2$ luôn tăng khi thêm biến, ngay cả khi biến đó không có ý nghĩa.

### 3.3 $R^2$ hiệu chỉnh

$$
\boxed{\bar{R}^2 = 1 - \frac{n-1}{n-p-1}(1 - R^2)}
$$

Phạt thêm biến ($p$ tăng → $n-p-1$ giảm → $\bar{R}^2$ giảm nếu biến mới không có đóng góp).

### 3.4 Kiểm định F (ý nghĩa tổng thể mô hình)

$$
H_0: \beta_1 = \beta_2 = \cdots = \beta_p = 0
\quad \text{vs} \quad
H_1: \exists\, j,\; \beta_j \neq 0
$$

$$
\boxed{F = \frac{\text{MSS}/p}{\text{RSS}/(n-p-1)}
= \frac{(\text{TSS}-\text{RSS})/p}{\text{RSS}/(n-p-1)}
\sim F_{p,\;n-p-1}}
$$

F lớn → bác bỏ $H_0$ → mô hình có ý nghĩa thống kê.

In [ ]:
metrics1 = model_metrics(y1, result1['y_hat'], result1['p'])
print_metrics(metrics1)
print(f"  y_mean = {metrics1['y_mean']:.6f}")
print(f"  df_model = {metrics1['df_model']},  df_resid = {metrics1['df_resid']}")

In [ ]:
y_np  = np.array(y1)
yh_np = np.array(result1['y_hat'])
n, p  = result1['n'], result1['p']

rss_np = np.sum((y_np - yh_np)**2)
tss_np = np.sum((y_np - y_np.mean())**2)
r2_np  = 1 - rss_np / tss_np
r2_adj_np = 1 - (n-1)/(n-p-1) * (1 - r2_np)
f_np   = ((tss_np - rss_np)/p) / (rss_np/(n-p-1))

print(f"NumPy  RSS={rss_np:.6f}  R²={r2_np:.6f}  R²_adj={r2_adj_np:.6f}  F={f_np:.6f}")
print(f"Ours   RSS={metrics1['rss']:.6f}  R2={metrics1['r2']:.6f}  R2_adj={metrics1['r2_adj']:.6f}  F={metrics1['f_stat']:.6f}")
print("Khop NumPy:", np.isclose(metrics1['rss'], rss_np, atol=1e-4)
      and np.isclose(metrics1['r2'], r2_np, atol=1e-4)
      and np.isclose(metrics1['f_stat'], f_np, atol=1e-4))

---
## Hàm 4 — `coef_inference`: Kiểm Định Hệ Số

### 4.1 Phân phối của $\hat{\boldsymbol{\beta}}$

Dưới giả thiết Gauss–Markov (GM1–GM5):

$$
\hat{\boldsymbol{\beta}} \sim \mathcal{N}\!\left(\boldsymbol{\beta},\; \sigma^2(\mathbf{X}^\top\mathbf{X})^{-1}\right)
$$

### 4.2 Standard Error (SE)

$$
\boxed{\text{SE}(\hat{\beta}_j) = \hat{\sigma}\,\sqrt{[(\mathbf{X}^\top\mathbf{X})^{-1}]_{jj}}}
$$

### 4.3 Kiểm định t (Student)

$$
H_0: \beta_j = 0 \quad \text{vs} \quad H_1: \beta_j \neq 0
$$

$$
\boxed{t_j = \frac{\hat{\beta}_j}{\text{SE}(\hat{\beta}_j)} \sim t_{n-p-1}}
\quad \text{(dưới } H_0\text{)}
$$

- **p-value** (hai phía): $p_j = 2\,P(T > |t_j|)$ với $T \sim t_{n-p-1}$
- Bác bỏ $H_0$ ở mức ý nghĩa $\alpha$ khi $p_j < \alpha$ (thường $\alpha = 0.05$)

### 4.4 Khoảng tin cậy $(1-\alpha)\cdot 100\%$

$$
\boxed{\hat{\beta}_j \pm t_{\alpha/2,\;n-p-1} \cdot \text{SE}(\hat{\beta}_j)}
$$

Với mức tin cậy 95%, $\alpha = 0.05$ và $t_{0.025, n-p-1}$ là phân vị của phân phối $t$.

### 4.5 Sơ đồ quy trình

```
X, y, β̂, σ̂²
     │
     ├─→ (XᵀX)⁻¹  ──→  SE(β̂ⱼ) = σ̂√[(XᵀX)⁻¹]ⱼⱼ
     │                              │
     │              ┌───────────────┴──────────────┐
     │              ↓                              ↓
     │         tⱼ = β̂ⱼ/SE(β̂ⱼ)          CI: β̂ⱼ ± t_crit·SE
     │              │
     │              ↓
     │         p-value = 2·P(T > |tⱼ|)
```

In [ ]:

inf1 = coef_inference(X1, y1, result1['beta_hat'], result1['sigma2'])
print_inference(inf1, feature_names=['x'], beta_hat=result1['beta_hat'])
print()
print(f"  t_critical (α/2=0.025, dof={inf1['dof']}): {inf1['t_crit']:.4f}")
print()
print("  Khoảng tin cậy 95%:")
names = ['intercept', 'x']
for j, name in enumerate(names):
    print(f"    {name:>10s}: [{inf1['ci_lower'][j]:+.4f},  {inf1['ci_upper'][j]:+.4f}]")

In [ ]:

X_d   = np.hstack([np.ones((len(X1),1)), np.array(X1)])
y_np  = np.array(y1)
n, k  = X_d.shape
dof   = n - k

beta_np = np.linalg.inv(X_d.T @ X_d) @ X_d.T @ y_np
yh_np   = X_d @ beta_np
rss_np  = np.sum((y_np - yh_np)**2)
s2_np   = rss_np / dof

se_np  = np.sqrt(s2_np * np.diag(np.linalg.inv(X_d.T @ X_d)))
t_np   = beta_np / se_np
pv_np  = 2 * (1 - stats.t.cdf(np.abs(t_np), dof))

print("        coef      SE      t-stat   p-value")
print("         (scipy reference)")
for j, name in enumerate(['intercept','x']):
    print(f"  {name:>10s}  {se_np[j]:.4f}  {t_np[j]:7.4f}  {pv_np[j]:.4f}")

print()
print("        coef      SE      t-stat   p-value")
print("         (cài đặt của ta)")
for j, name in enumerate(['intercept','x']):
    print(f"  {name:>10s}  {inf1['se'][j]:.4f}  {inf1['t_stats'][j]:7.4f}  {inf1['p_values'][j]:.4f}")

print()
print("Khop SciPy (se, p-value):",
      np.allclose(inf1['se'], se_np, atol=1e-4),
      np.allclose(inf1['p_values'], pv_np, atol=1e-4))

---
## Tổng Kết

| Hàm | Công thức chính | Kết quả kiểm chứng |
|-----|----------------|--------------------|
| `ols_fit` | $\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$, $\hat{\sigma}^2 = \text{RSS}/(n-p-1)$ | Khớp NumPy/sklearn|
| `hat_matrix` | $H = \mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top$, $H^2=H$ | Idempotent, đối xứng  |
| `model_metrics` | $R^2 = 1 - \text{RSS/TSS}$, $\bar{R}^2$, $F$-stat | Khớp NumPy |
| `coef_inference` | $t_j = \hat{\beta}_j/\text{SE}_j$, CI $= \hat{\beta}_j \pm t_{\alpha/2}\cdot\text{SE}_j$ | Khớp scipy.stats |